In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import Row
from databricks.sdk.runtime import dbutils


In [2]:
spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
    )

In [3]:
dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")

print(f"Using catalog: {catalog}")

Using catalog: workspace


/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [4]:
bronze_orders = spark.table(f"{catalog}.bronze.orders")

bronze_orders.printSchema()
print("Bronze orders: ", bronze_orders.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Bronze orders:  99441


In [5]:
silver_orders_base = (
    bronze_orders
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )
)

silver_orders_base.select(
    "order_id",
    "order_status",
    "_source_file",
    "_ingested_at"
).show(10, truncate=False)

+--------------------------------+------------+-----------------------------------------------------------------+--------------------------+
|order_id                        |order_status|_source_file                                                     |_ingested_at              |
+--------------------------------+------------+-----------------------------------------------------------------+--------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-11 11:36:35.931389|
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-11 11:36:35.931389|
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-11 11:36:35.931389|
|949d5b44dbf5de918fe9c16f97b45f8a|delivered   |dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-11 11:36:35.931389|
|ad21c59c0840

In [6]:
silver_orders_metrics = (
    silver_orders_base
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date")
        )
    )
    .withColumn(
        "is_late",
        F.when(
            F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_estimated_delivery_date").isNotNull(),
            F.col("order_delivered_customer_date")
            > F.col("order_estimated_delivery_date")
        ).otherwise(F.lit(None).cast("boolean"))
    )
)

In [7]:
silver_orders_metrics.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "delivery_delay_days",
    "is_late"
).filter(
    F.col("order_delivered_customer_date").isNotNull()
).show(10, truncate=False)

+--------------------------------+------------+------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|order_id                        |order_status|order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|
+--------------------------------+------------+------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |2017-10-02 10:56:33     |2017-10-10 21:25:13          |2017-10-18 00:00:00          |8            |-8                 |false  |
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |2018-07-24 20:41:37     |2018-08-07 15:27:45          |2018-08-13 00:00:00          |14           |-6                 |false  |
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |2018-08-08 08:38:49     |2018-08-17 18:06:29          |2018-09-04 00:00:00    

In [8]:
dq_summary = silver_orders_metrics.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("null_order_id"),

    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("null_customer_id"),

    F.sum(
        F.when(
            F.col("order_purchase_timestamp").isNull(),
            1
        ).otherwise(0)
    ).alias("null_purchase_timestamp"),

    F.sum(
        F.when(
            F.col("order_delivered_customer_date")
            < F.col("order_purchase_timestamp"),
            1
        ).otherwise(0)
    ).alias("delivery_before_purchase"),

    F.sum(
        F.when(
            (F.col("order_status") == "delivered")
            & F.col("order_delivered_customer_date").isNull(),
            1
        ).otherwise(0)
    ).alias("delivered_status_missing_delivery_date")
)
dq_summary.show(truncate=False)

+----------+-------------+----------------+-----------------------+------------------------+--------------------------------------+
|total_rows|null_order_id|null_customer_id|null_purchase_timestamp|delivery_before_purchase|delivered_status_missing_delivery_date|
+----------+-------------+----------------+-----------------------+------------------------+--------------------------------------+
|99441     |0            |0               |0                      |0                       |8                                     |
+----------+-------------+----------------+-----------------------+------------------------+--------------------------------------+



In [9]:
silver_orders_checked = (
    silver_orders_metrics
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("order_id").isNull(),
            F.lit("missing_order_id")
        )
        .when(
            F.col("customer_id").isNull(),
            F.lit("missing_customer_id")
        )
        .when(
            F.col("order_purchase_timestamp").isNull(),
            F.lit("missing_purchase_timestamp")
        )
        .when(
            F.col("order_delivered_customer_date")
            < F.col("order_purchase_timestamp"),
            F.lit("delivery_before_purchase")
        )
        .when(
            (F.col("order_status") == "delivered")
            & F.col("order_delivered_customer_date").isNull(),
            F.lit("delivered_status_missing_delivery_date")
        )
        .otherwise(F.lit(None).cast("string"))
    )
)

In [10]:
silver_orders_checked.groupBy(
    "rejection_reason"
).count().orderBy(
    F.col("count").desc()
).show(truncate=False)

+--------------------------------------+-----+
|rejection_reason                      |count|
+--------------------------------------+-----+
|NULL                                  |99433|
|delivered_status_missing_delivery_date|8    |
+--------------------------------------+-----+



In [11]:
silver_orders = (
    silver_orders_checked
    .filter(F.col("rejection_reason").isNull())
    .drop("rejection_reason")
)

rejected_orders = (
    silver_orders_checked
    .filter(F.col("rejection_reason").isNotNull())
)

In [12]:
print("Silver orders:", silver_orders.count())
print("Rejected orders:", rejected_orders.count())
print("Total:", silver_orders.count() + rejected_orders.count())

Silver orders: 99433
Rejected orders: 8
Total: 99441


In [13]:
(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.orders")
)

In [14]:
spark.sql(f"""
    DESCRIBE DETAIL {catalog}.silver.orders
""").select(
    "format",
    "name",
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

print(
    "Persisted Silver rows:",
    spark.table(f"{catalog}.silver.orders").count()
)

+------+-----------------------+--------+-----------+
|format|name                   |numFiles|sizeInBytes|
+------+-----------------------+--------+-----------+
|delta |workspace.silver.orders|1       |5869448    |
+------+-----------------------+--------+-----------+

Persisted Silver rows: 99433


In [15]:
(
    rejected_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.rejected_orders")
)

In [16]:
rejected_orders_saved = spark.table(
    f"{catalog}.silver.rejected_orders"
)

print("Rejected rows:", rejected_orders_saved.count())

print(rejected_orders_saved.select(
    "order_id",
    "order_status",
    "order_delivered_customer_date",
    "rejection_reason"
).show(truncate=False))

Rejected rows: 8
+--------------------------------+------------+-----------------------------+--------------------------------------+
|order_id                        |order_status|order_delivered_customer_date|rejection_reason                      |
+--------------------------------+------------+-----------------------------+--------------------------------------+
|2d1e2d5bf4dc7227b3bfebb81328c15f|delivered   |NULL                         |delivered_status_missing_delivery_date|
|f5dd62b788049ad9fc0526e3ad11a097|delivered   |NULL                         |delivered_status_missing_delivery_date|
|2ebdfc4f15f23b91474edf87475f108e|delivered   |NULL                         |delivered_status_missing_delivery_date|
|e69f75a717d64fc5ecdfae42b2e8e086|delivered   |NULL                         |delivered_status_missing_delivery_date|
|0d3268bad9b086af767785e3f0fc0133|delivered   |NULL                         |delivered_status_missing_delivery_date|
|2d858f451373b04fb5c984a1cc2defaf|delivered   |

In [17]:
bronze_customers = spark.table(f"{catalog}.bronze.customers")

bronze_customers.printSchema()
print("Bronze customers:", bronze_customers.count())

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Bronze customers: 99441


In [18]:
silver_customers_base = (
    bronze_customers
    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "customer_state",
        F.upper(F.trim(F.col("customer_state")))
    )
)

In [19]:
silver_customers_base.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "_source_file",
    "_ingested_at"
).show(10, truncate=False)

+--------------------------------+--------------------------------+---------------------+--------------+--------------------------------------------------------------------+--------------------------+
|customer_id                     |customer_unique_id              |customer_city        |customer_state|_source_file                                                        |_ingested_at              |
+--------------------------------+--------------------------------+---------------------+--------------+--------------------------------------------------------------------+--------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|franca               |SP            |dbfs:/Volumes/workspace/bronze/raw_files/olist_customers_dataset.csv|2026-08-11 10:52:08.199485|
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|sao bernardo do campo|SP            |dbfs:/Volumes/workspace/bronze/raw_files/olist_customers_dataset.csv|2026-08-11 10:52:08.199

In [20]:
customer_dq_summary = silver_customers_base.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("null_customer_id"),

    F.sum(
        F.when(F.col("customer_unique_id").isNull(), 1).otherwise(0)
    ).alias("null_customer_unique_id"),

    F.sum(
        F.when(F.col("customer_zip_code_prefix").isNull(), 1).otherwise(0)
    ).alias("null_zip_code"),

    F.sum(
        F.when(F.col("customer_city").isNull(), 1).otherwise(0)
    ).alias("null_city"),

    F.sum(
        F.when(F.col("customer_state").isNull(), 1).otherwise(0)
    ).alias("null_state")
)

In [21]:
customer_dq_summary.show(truncate=False)

+----------+----------------+-----------------------+-------------+---------+----------+
|total_rows|null_customer_id|null_customer_unique_id|null_zip_code|null_city|null_state|
+----------+----------------+-----------------------+-------------+---------+----------+
|99441     |0               |0                      |0            |0        |0         |
+----------+----------------+-----------------------+-------------+---------+----------+



In [22]:
duplicate_customer_ids = (
    silver_customers_base
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer_id values:", duplicate_customer_ids.count())

Duplicate customer_id values: 0


In [23]:
(
    silver_customers_base.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.customers")
)

In [24]:
spark.sql(f"""
    DESCRIBE DETAIL {catalog}.silver.customers
""").select(
    "format",
    "name",
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

print(
    "Persisted Silver customers:",
    spark.table(f"{catalog}.silver.customers").count()
)

+------+--------------------------+--------+-----------+
|format|name                      |numFiles|sizeInBytes|
+------+--------------------------+--------+-----------+
|delta |workspace.silver.customers|1       |3740554    |
+------+--------------------------+--------+-----------+

Persisted Silver customers: 99441


In [25]:
bronze_order_items = spark.table(f"{catalog}.bronze.order_items")
bronze_order_items.printSchema()

print("Bronze order items: ", bronze_order_items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Bronze order items:  112650


In [26]:
bronze_order_items.show(5, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+----------------------------------------------------------------------+------------------------+--------------------------+--------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|_source_file                                                          |_source_file_modified_at|_ingested_at              |_source_entity|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+----------------------------------------------------------------------+------------------------+--------------------------+--------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017

In [27]:
silver_order_items_base = (
    bronze_order_items
    .withColumn(
        "item_total",
        F.col("price") + F.col("freight_value")
    )
)

In [28]:
silver_order_items_base.select(
    "order_id",
    "order_item_id",
    "product_id",
    "price",
    "freight_value",
    "item_total"
).show(10, truncate=False)

+--------------------------------+-------------+--------------------------------+------+-------------+----------+
|order_id                        |order_item_id|product_id                      |price |freight_value|item_total|
+--------------------------------+-------------+--------------------------------+------+-------------+----------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|58.90 |13.29        |72.19     |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|239.90|19.93        |259.83    |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|199.00|17.87        |216.87    |
|00024acbcdf0a6daa1e931b038114c75|1            |7634da152a4610f1595efa32f14722fc|12.99 |12.79        |25.78     |
|00042b26cf59d7ce69dfabb4e55b4fd9|1            |ac6c3623068f30de03045865e4e10089|199.90|18.14        |218.04    |
|00048cc3ae777c65dbb7d2a0634bc1ea|1            |ef92defde845ab8450f9d70c526ef70f|21.90 |

In [29]:
order_items_dq_summary = (
    silver_order_items_base
    .agg(
        F.count("*").alias("total_rows"),
        F.sum(
            F.when(F.col("order_id").isNull(), 1).otherwise(0)
        ).alias("null_order_id"),
        F.sum(
            F.when(F.col("order_item_id").isNull(), 1).otherwise(0)
        ).alias("null_order_item_id"),
        F.sum(
            F.when(F.col("product_id").isNull(), 1).otherwise(0)
        ).alias("null_product_id"),
        F.sum(
            F.when(F.col("price") < 0, 1).otherwise(0)
        ).alias("negative_price"),
        F.sum(
            F.when(F.col("freight_value") < 0, 1).otherwise(0)
        ).alias("negative_freight"),
    )
)

order_items_dq_summary.show(truncate=False)

+----------+-------------+------------------+---------------+--------------+----------------+
|total_rows|null_order_id|null_order_item_id|null_product_id|negative_price|negative_freight|
+----------+-------------+------------------+---------------+--------------+----------------+
|112650    |0            |0                 |0              |0             |0               |
+----------+-------------+------------------+---------------+--------------+----------------+



In [30]:
duplicate_order_items = (
    silver_order_items_base
    .groupBy(
        "order_id",
        "order_item_id"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate (order_id, order_item_id) keys:",
    duplicate_order_items.count()
)

Duplicate (order_id, order_item_id) keys: 0


In [31]:
silver_order_ref = spark.table(f"{catalog}.silver.orders")

unmatched_order_items = (
    silver_order_items_base
    .join(silver_order_ref, on="order_id", how="left_anti")
)

print(
    "Order items with no valid Silver order:",
    unmatched_order_items.count()
)

Order items with no valid Silver order: 8


In [32]:
rejected_orders_ref = spark.table(f"{catalog}.silver.rejected_orders")

unmatched_vs_rejected = (
    unmatched_order_items
    .select("order_id")
    .distinct()
    .join(
        rejected_orders_ref.select("order_id", "rejection_reason"),
        on="order_id",
        how="left"
    )
)

unmatched_vs_rejected.show(truncate=False)

+--------------------------------+--------------------------------------+
|order_id                        |rejection_reason                      |
+--------------------------------+--------------------------------------+
|20edc82cf5400ce95e1afacc25798b31|delivered_status_missing_delivery_date|
|2ebdfc4f15f23b91474edf87475f108e|delivered_status_missing_delivery_date|
|ab7c89dc1bf4a1ead9d6ec1ec8968a84|delivered_status_missing_delivery_date|
|2d858f451373b04fb5c984a1cc2defaf|delivered_status_missing_delivery_date|
|0d3268bad9b086af767785e3f0fc0133|delivered_status_missing_delivery_date|
|e69f75a717d64fc5ecdfae42b2e8e086|delivered_status_missing_delivery_date|
|f5dd62b788049ad9fc0526e3ad11a097|delivered_status_missing_delivery_date|
|2d1e2d5bf4dc7227b3bfebb81328c15f|delivered_status_missing_delivery_date|
+--------------------------------+--------------------------------------+



In [33]:
silver_order_items = (
    silver_order_items_base
    .join(
        silver_order_ref.select("order_id"),
        on="order_id",
        how="left_semi"
    )
)

rejected_order_items = (
    silver_order_items_base
    .join(
        silver_order_ref.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .withColumn(
        "rejection_reason",
        F.lit("parent_order_rejected")
    )
)

In [34]:
print("Silver order items:", silver_order_items.count())
print("Rejected order items:", rejected_order_items.count())
print(
    "Total:",
    silver_order_items.count() + rejected_order_items.count()
)

Silver order items: 112642
Rejected order items: 8
Total: 112650


In [35]:
(
    silver_order_items.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.order_items")
)

In [36]:
(
    rejected_order_items.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.rejected_order_items")
)

In [37]:
print(
    "Persisted Silver order items:",
    spark.table(f"{catalog}.silver.order_items").count()
)

print(
    "Persisted rejected order items:",
    spark.table(f"{catalog}.silver.rejected_order_items").count()
)

Persisted Silver order items: 112642
Persisted rejected order items: 8


In [38]:
bronze_products = spark.table(f"{catalog}.bronze.products")

bronze_products.printSchema()
print("Bronze products:", bronze_products.count())

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Bronze products: 32951


In [39]:
silver_products_base = (
    bronze_products
    .withColumnRenamed(
        "product_name_lenght",
        "product_name_length"
    )
    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )
    .withColumn(
        "product_category_name",
        F.when(
            F.col("product_category_name").isNull(),
            F.lit("unknown")
        ).otherwise(
            F.lower(
                F.trim(F.col("product_category_name"))
            )
        )
    )
)

In [40]:
silver_products_base.select(
    "product_id",
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty"
).show(10, truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+
|product_id                      |product_category_name|product_name_length|product_description_length|product_photos_qty|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                 |287                       |1                 |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                 |276                       |1                 |
|96bd76ec8810374ed1b65e291975717f|esporte_lazer        |46                 |250                       |1                 |
|cef67bcfe19066a932b7673e239eb23d|bebes                |27                 |261                       |1                 |
|9dc1a7de274444849c219cff195d0b71|utilidades_domesticas|37                 |402                       |4                 |
|41d3672d4792049

In [41]:
products_dq_summary = silver_products_base.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("product_id").isNull(), 1).otherwise(0)
    ).alias("null_product_id"),

    F.sum(
        F.when(F.col("product_weight_g") < 0, 1).otherwise(0)
    ).alias("negative_weight"),

    F.sum(
        F.when(F.col("product_length_cm") < 0, 1).otherwise(0)
    ).alias("negative_length"),

    F.sum(
        F.when(F.col("product_height_cm") < 0, 1).otherwise(0)
    ).alias("negative_height"),

    F.sum(
        F.when(F.col("product_width_cm") < 0, 1).otherwise(0)
    ).alias("negative_width"),

    F.sum(
        F.when(F.col("product_photos_qty") < 0, 1).otherwise(0)
    ).alias("negative_photos_qty")
)

In [42]:
products_dq_summary.show(truncate=False)

+----------+---------------+---------------+---------------+---------------+--------------+-------------------+
|total_rows|null_product_id|negative_weight|negative_length|negative_height|negative_width|negative_photos_qty|
+----------+---------------+---------------+---------------+---------------+--------------+-------------------+
|32951     |0              |0              |0              |0              |0             |0                  |
+----------+---------------+---------------+---------------+---------------+--------------+-------------------+



In [43]:
duplicate_product_ids = (
    silver_products_base
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate product_id values:",
    duplicate_product_ids.count()
)

Duplicate product_id values: 0


In [44]:
silver_order_items_ref = spark.table(f"{catalog}.silver.order_items")

In [45]:
unmatched_products = (
    silver_order_items_ref
    .join(
        F.broadcast(
            silver_products_base.select("product_id")
        ),
        on="product_id",
        how="left_anti"
    )
)

print(
    "Silver order items with no matching product:",
    unmatched_products.count()
)

Silver order items with no matching product: 0


In [46]:
(
    silver_products_base.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.products")
)

In [47]:
spark.sql(f"""
    DESCRIBE DETAIL {catalog}.silver.products
""").select(
    "format",
    "name",
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

print(
    "Persisted Silver products:",
    spark.table(f"{catalog}.silver.products").count()
)

+------+-------------------------+--------+-----------+
|format|name                     |numFiles|sizeInBytes|
+------+-------------------------+--------+-----------+
|delta |workspace.silver.products|1       |795041     |
+------+-------------------------+--------+-----------+

Persisted Silver products: 32951


In [48]:
bronze_payments = spark.table(f"{catalog}.bronze.payments")

bronze_payments.printSchema()
print("Bronze payments:", bronze_payments.count())

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(10,2) (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Bronze payments: 103886


In [49]:
bronze_payments.show(2, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+-------------------------------------------------------------------------+------------------------+--------------------------+--------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|_source_file                                                             |_source_file_modified_at|_ingested_at              |_source_entity|
+--------------------------------+------------------+------------+--------------------+-------------+-------------------------------------------------------------------------+------------------------+--------------------------+--------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |dbfs:/Volumes/workspace/bronze/raw_files/olist_order_payments_dataset.csv|2026-08-11 13:21:03     |2026-08-11 13:23:51.396453|payments      |
|a9810da82917af2d9aefd1278f1

In [50]:
silver_payments_base = (
    bronze_payments
    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type")))
    )
)

In [51]:
payments_dq_summary = silver_payments_base.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("null_order_id"),

    F.sum(
        F.when(F.col("payment_sequential").isNull(), 1).otherwise(0)
    ).alias("null_payment_sequential"),

    F.sum(
        F.when(F.col("payment_type").isNull(), 1).otherwise(0)
    ).alias("null_payment_type"),

    F.sum(
        F.when(F.col("payment_value") < 0, 1).otherwise(0)
    ).alias("negative_payment_value"),

    F.sum(
        F.when(F.col("payment_installments") < 0, 1).otherwise(0)
    ).alias("negative_installments")
)

In [52]:
payments_dq_summary.show(truncate=False)

+----------+-------------+-----------------------+-----------------+----------------------+---------------------+
|total_rows|null_order_id|null_payment_sequential|null_payment_type|negative_payment_value|negative_installments|
+----------+-------------+-----------------------+-----------------+----------------------+---------------------+
|103886    |0            |0                      |0                |0                     |0                    |
+----------+-------------+-----------------------+-----------------+----------------------+---------------------+



In [53]:
duplicate_payments = (
    silver_payments_base
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate payment keys:",
    duplicate_payments.count()
)

Duplicate payment keys: 0


In [54]:
silver_orders_ref = spark.table(f"{catalog}.silver.orders")

unmatched_payments = (
    silver_payments_base
    .join(
        F.broadcast(
            silver_orders_ref.select("order_id")
        ),
        on="order_id",
        how="left_anti"
    )
)

print(
    "Payment rows with no matching Silver order:",
    unmatched_payments.count()
)

Payment rows with no matching Silver order: 8


In [55]:
rejected_orders_ref = spark.table(f"{catalog}.silver.rejected_orders")

unmatched_payment_reasons = (
    unmatched_payments
    .join(
        rejected_orders_ref.select(
            "order_id",
            "rejection_reason"
        ),
        on="order_id",
        how="left"
    )
)

unmatched_payment_reasons.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_value",
    "rejection_reason"
).show(20, truncate=False)

+--------------------------------+------------------+------------+-------------+--------------------------------------+
|order_id                        |payment_sequential|payment_type|payment_value|rejection_reason                      |
+--------------------------------+------------------+------------+-------------+--------------------------------------+
|0d3268bad9b086af767785e3f0fc0133|1                 |credit_card |204.62       |delivered_status_missing_delivery_date|
|20edc82cf5400ce95e1afacc25798b31|1                 |credit_card |54.97        |delivered_status_missing_delivery_date|
|2d1e2d5bf4dc7227b3bfebb81328c15f|1                 |credit_card |134.83       |delivered_status_missing_delivery_date|
|2ebdfc4f15f23b91474edf87475f108e|1                 |credit_card |158.07       |delivered_status_missing_delivery_date|
|2d858f451373b04fb5c984a1cc2defaf|1                 |credit_card |194.00       |delivered_status_missing_delivery_date|
|e69f75a717d64fc5ecdfae42b2e8e086|1     

In [56]:
silver_payments = (
    silver_payments_base
    .join(
        silver_orders_ref.select("order_id"),
        on="order_id",
        how="left_semi"
    )
)

rejected_payments = (
    silver_payments_base
    .join(
        silver_orders_ref.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .withColumn(
        "rejection_reason",
        F.lit("parent_order_rejected")
    )
)

print("Silver payments:", silver_payments.count())
print("Rejected payments:", rejected_payments.count())
print("Total:", silver_payments.count() + rejected_payments.count())

Silver payments: 103878
Rejected payments: 8
Total: 103886


In [57]:
(
    silver_payments.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.payments")
)

(
    rejected_payments.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.rejected_payments")
)

In [58]:
print(
    "Persisted Silver payments:",
    spark.table(f"{catalog}.silver.payments").count()
)

print(
    "Persisted rejected payments:",
    spark.table(f"{catalog}.silver.rejected_payments").count()
)

Persisted Silver payments: 103878
Persisted rejected payments: 8


In [59]:
silver_orders_ref = spark.table(f"{catalog}.silver.orders")
silver_customers_ref = spark.table(f"{catalog}.silver.customers")

unmatched_customers = (
    silver_orders_ref
    .join(
        F.broadcast(
            silver_customers_ref.select("customer_id")
        ),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Silver orders with no matching customer:",
    unmatched_customers.count()
)

Silver orders with no matching customer: 0


In [60]:
tables = [
    f"{catalog}.silver.orders",
    f"{catalog}.silver.customers",
    f"{catalog}.silver.order_items",
    f"{catalog}.silver.products",
    f"{catalog}.silver.payments",
    f"{catalog}.silver.rejected_orders",
    f"{catalog}.silver.rejected_order_items",
    f"{catalog}.silver.rejected_payments",
]

for table in tables:
    print(f"{table}: {spark.table(table).count()}")

workspace.silver.orders: 99433
workspace.silver.customers: 99441
workspace.silver.order_items: 112642
workspace.silver.products: 32951
workspace.silver.payments: 103878
workspace.silver.rejected_orders: 8
workspace.silver.rejected_order_items: 8
workspace.silver.rejected_payments: 8


In [61]:
silver_validation_summary = spark.createDataFrame([
    Row(table_name="orders", row_count=99433, rejected_count=8),
    Row(table_name="customers", row_count=99441, rejected_count=0),
    Row(table_name="order_items", row_count=112642, rejected_count=8),
    Row(table_name="products", row_count=32951, rejected_count=0),
    Row(table_name="payments", row_count=103878, rejected_count=8),
])

print(silver_validation_summary.show(truncate=False))

+-----------+---------+--------------+
|table_name |row_count|rejected_count|
+-----------+---------+--------------+
|orders     |99433    |8             |
|customers  |99441    |0             |
|order_items|112642   |8             |
|products   |32951    |0             |
|payments   |103878   |8             |
+-----------+---------+--------------+

None


In [62]:
reject_tables = [
    f"{catalog}.silver.rejected_orders",
    f"{catalog}.silver.rejected_order_items",
    f"{catalog}.silver.rejected_payments",
]

for table in reject_tables:
    print(f"\n{table}")
    (
        spark.table(table)
        .groupBy("rejection_reason")
        .count()
        .show(truncate=False)
    )


workspace.silver.rejected_orders
+--------------------------------------+-----+
|rejection_reason                      |count|
+--------------------------------------+-----+
|delivered_status_missing_delivery_date|8    |
+--------------------------------------+-----+


workspace.silver.rejected_order_items
+---------------------+-----+
|rejection_reason     |count|
+---------------------+-----+
|parent_order_rejected|8    |
+---------------------+-----+


workspace.silver.rejected_payments
+---------------------+-----+
|rejection_reason     |count|
+---------------------+-----+
|parent_order_rejected|8    |
+---------------------+-----+



In [63]:
spark.sql(f"""
SHOW TABLES IN {catalog}.silver
""").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|silver  |customers           |false      |
|silver  |order_items         |false      |
|silver  |orders              |false      |
|silver  |payments            |false      |
|silver  |products            |false      |
|silver  |rejected_order_items|false      |
|silver  |rejected_orders     |false      |
|silver  |rejected_payments   |false      |
+--------+--------------------+-----------+



In [64]:
silver_tables = [
    f"{catalog}.silver.orders",
    f"{catalog}.silver.customers",
    f"{catalog}.silver.order_items",
    f"{catalog}.silver.products",
    f"{catalog}.silver.payments"
]

for table in silver_tables:
    print(f"\n{table}")
    spark.sql(f"""
    DESCRIBE HISTORY {table}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).show(truncate=False)


workspace.silver.orders
+-------+-------------------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationMetrics                                                                                                                                    |
+-------+-------------------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------+
|3      |2026-08-12 08:49:52|CREATE OR REPLACE TABLE AS SELECT|{numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 5869448, numDeletionVectorsRemoved -> 0, numOutputRows -> 99433, numOutputBytes -> 5869448}|
|2      |2026-08-12 06:43:11|CREATE OR REPLACE TABLE AS SELECT|{numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 586

In [65]:
orders_v0 = spark.sql(f"""
    SELECT *
    FROM {catalog}.silver.orders VERSION AS OF 0
""")

orders_current = spark.table(f"{catalog}.silver.orders")

print("Orders version 0:", orders_v0.count())
print("Orders current:", orders_current.count())

Orders version 0: 99433
Orders current: 99433


In [66]:
orders_v0 = spark.sql(f"""
    SELECT *
    FROM {catalog}.silver.orders VERSION AS OF 0
""")

orders_current = spark.table(f"{catalog}.silver.orders")

v0_not_current = orders_v0.exceptAll(orders_current).count()
current_not_v0 = orders_current.exceptAll(orders_v0).count()

print("Rows in v0 but not current:", v0_not_current)
print("Rows in current but not v0:", current_not_v0)

Rows in v0 but not current: 0
Rows in current but not v0: 0
